# Mahalakshmi: SASRec+ on Amazon Baby

Use the team's shared data loader and original-style HR/NDCG evaluator. Start with **smoke** mode, then switch to **full**. Smoke results only show that the pipeline runs.

Before running: import this notebook into Kaggle; attach the private input containing `cmpe257_kaggle_code.zip` and `dataset.pkl`; enable Internet for dependencies and a GPU for full training. This notebook uses one GPU. No GitHub token or push is needed.

Read `docs/KAGGLE_TRAINING.md` in the repo for the full walkthrough. The notebook launches Python modules from the attached source ZIP; the notebook alone is not enough.


## 1. Load the shared code and dataset paths


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# Attach the PRIVATE Kaggle dataset containing these two files.
# Leave as None to discover a single matching file; set explicit paths if needed.
CODE_ZIP = None  # e.g. '/kaggle/input/cmpe257-training/cmpe257_kaggle_code.zip'
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATA_FILE = None  # e.g. '/kaggle/input/cmpe257-training/dataset.pkl'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

DATASET_PATH = attached_file('dataset.pkl', DATA_FILE)
REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
print('Dataset:', DATASET_PATH)


## 2. Install dependencies
If Kaggle requests a session restart after installation, restart and rerun from cell 1. Do not change package versions independently without recording the change.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Import every required model now, so dependency errors happen before training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 3. Choose a run
Keep `RUN_MODE = "smoke"` for the first test. For a real experiment change it to `"full"`, choose a new `RUN_NAME`, and keep the agreed shared settings. `FINALIZE = False` saves progress without test evaluation when splitting training across sessions.


In [ ]:
RUN_MODE = 'smoke'  # 'full' uses all 11,761 users, hidden size 128, up to 500 epochs
RUN_NAME = 'mahalakshmi_sasrec_' + RUN_MODE + '_01'
FINALIZE = True
RESUME_CHECKPOINT = None  # Path to this model's latest.pt from a saved previous run
cfg = make_config('sasrec', RUN_MODE)
# Example only, after agreeing with the team: cfg['train_batch_size'] = 128
# Keep evaluation_batch_size=512 for full runs, even if training batch size changes.
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
print(json.dumps(cfg, indent=2))
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print(dataset_summary(data))


## 4. Train SASRec+
This uses `SASRec` from the authors' vendored code, including position embeddings and per-token cross-entropy training. It is the team's SASRec+ baseline.

Validation selects the checkpoint. Test evaluation runs only at the end when `FINALIZE = True`.


In [ ]:
result = train(cfg, DATASET_PATH, RUN_DIR / 'sasrec',
               resume=RESUME_CHECKPOINT, finalize=FINALIZE)
MODEL_DIR = RUN_DIR / 'sasrec'


## Inspect the results
HR and NDCG are percentages at 5, 10, and 20. The shared evaluator includes the original candidate columns and averages batch means like upstream. Do not compare smoke results with full results. Test results should be used for final reporting, not tuning settings.


In [ ]:
print(json.dumps(result, indent=2))
print('Saved files:', [p.name for p in MODEL_DIR.iterdir()])


## Try inference from the saved best model
This example uses only a user's visible validation history. Product IDs are shown because this dataset has no names/images. This demo hides padding ID 0; the benchmark evaluator keeps its upstream candidate rules. Previously seen items remain eligible.


In [ ]:
model, saved_cfg = load_for_inference(MODEL_DIR / 'best.pt')
example = SequenceDataset(data, 'val', saved_cfg['max_len'])[0]
seed_everything(saved_cfg['seed'])
scores = predict_scores(model, [example['input_ids']]).cpu()
scores[:, 0] = -torch.inf  # Demo only: do not recommend padding
recommended_ids = torch.topk(scores, k=10, dim=-1).indices[0].tolist()
print('Visible history:', [i for i in example['input_ids'] if i])
print('Recommended product IDs:', recommended_ids)


## Save your work and resume later
Download `best.pt` for inference and `latest.pt` for resuming, plus `config.json`, `environment.json`, `requirements-lock.txt`, `history.json`, and `results.json`. Simul should also keep the `pretrain/` files. Keep weights outside Git; share small result/config files with the team.

Use Kaggle **Save Version** to preserve notebook outputs. **Save & Run All** reruns cells from the top in a fresh session; it is a new training execution. Download or save the output before ending a session—an in-memory checkpoint cannot survive a lost session.

To resume: attach the saved output, use the identical configuration and run mode, set `RESUME_CHECKPOINT` to that model's `latest.pt`, and run into a new writable output directory. You can increase `cfg['epochs']`; other training settings must match. `FINALIZE = False` avoids testing intermediate chunks. An early-stopped run stays stopped. Resume is at the last completed epoch, not the middle of an epoch. A changed GPU/library environment may change numerical results.

If training runs out of GPU memory, start a new run with a smaller **training** batch size and record it. Keep the agreed evaluation batch size unchanged. See the guide for limitations.
